# Loop 1 — find and fix the bug in a student's submission

Takes a student's failing attempt and the reference solution, and repairs the code. The
student never sees the repaired code — it goes to loop 2, which turns it into a hint.

Assignment checklist:

* **tools** — `fetch_best_submission`, `propose_fix`, `submit_to_platform` (all with enums
  and required fields)
* **loop** — observe (fetch) -> reason (diagnose) -> act (propose) -> verify (run the tests)
* **stops** — tests pass, step cap, or the same patch proposed twice
* **gate** — `submit_to_platform` asks in chat; a judge submission cannot be unsent
* **error branch** — a broken test runner returns `ok=False` and does not count as a
  failing test

In [ ]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

STATE = Path("state")
STATE.mkdir(exist_ok=True)

OFFLINE = not os.getenv("LLM_API_KEY")
# The SDK refuses an empty key, and everything below the live cell runs without one.
client = OpenAI(base_url=os.getenv("LLM_BASE_URL"), api_key=os.getenv("LLM_API_KEY") or "offline")
MODEL = os.getenv("MODEL_FIXER", "anthropic/claude-sonnet-5")

## The problem and the student's attempt

Print the largest sum of a window of exactly `k` elements. The student's loop stops one
iteration early, so it never scores the last window — it passes the tests where the answer
sits early in the array and fails the rest.

In [ ]:
BUGGY = '''import sys
data = sys.stdin.read().split()
n, k = int(data[0]), int(data[1])
a = [int(x) for x in data[2:2 + n]]
window = sum(a[:k])
best = window
for i in range(1, n - k):
    window += a[i + k - 1] - a[i - 1]
    best = max(best, window)
print(best)
'''

REFERENCE = BUGGY.replace("range(1, n - k)", "range(1, n - k + 1)")

TESTS = [
    ("5 2\n5 4 3 2 1\n", "9"),
    ("6 3\n1 9 1 1 1 1\n", "11"),
    ("4 2\n1 1 1 5\n", "6"),
    ("3 1\n1 2 3\n", "3"),
    ("2 2\n7 8\n", "15"),
]

SUBMISSIONS = [
    {"id": "s100", "at": "2026-07-20T10:00Z", "passed": 1, "code": "print(0)\n"},
    {"id": "s101", "at": "2026-07-20T11:30Z", "passed": 3, "code": BUGGY},
    {"id": "s102", "at": "2026-07-20T12:05Z", "passed": 3, "code": BUGGY + "# tweak\n"},
]

## Tools

`mistake_tag` is an enum rather than free text because we count them: the tally per student
is what we feed into the next session's prompt.

In [ ]:
MISTAKE_TAGS = ["off_by_one", "uninitialized", "overflow", "wrong_type",
                "missing_edge_case", "io_format", "wrong_algorithm"]

TOOLS = [
    {"type": "function", "function": {
        "name": "fetch_best_submission",
        "description": "Get the student's best attempt at this problem. Call this once at the "
                       "start. Do not call it again — the history does not change during a "
                       "session and a second call just wastes a step.",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {"type": "string"},
                "problem_id": {"type": "string"},
                "platform": {"type": "string", "enum": ["codeforces", "ejudge", "yandex_contest"]},
            },
            "required": ["user_id", "problem_id", "platform"],
        },
    }},
    {"type": "function", "function": {
        "name": "propose_fix",
        "description": "Submit a corrected version of the program for automated testing. Call "
                       "it once you can name a concrete defect. Do not use it to try random "
                       "changes, do not rewrite the solution from scratch, and do not resend "
                       "code you already sent.",
        "parameters": {
            "type": "object",
            "properties": {
                "diagnosis": {"type": "string", "description": "One sentence: what is wrong."},
                "mistake_tag": {"type": "string", "enum": MISTAKE_TAGS},
                "fixed_code": {"type": "string", "description": "The whole file, not a diff."},
            },
            "required": ["diagnosis", "mistake_tag", "fixed_code"],
        },
    }},
    {"type": "function", "function": {
        "name": "submit_to_platform",
        "description": "Send the repaired code to the judge under our service account. This is "
                       "public, permanent and rate-limited, so it needs a human to confirm. "
                       "Call it only after the tests already passed.",
        "parameters": {
            "type": "object",
            "properties": {
                "problem_id": {"type": "string"},
                "platform": {"type": "string", "enum": ["codeforces", "ejudge", "yandex_contest"]},
            },
            "required": ["problem_id", "platform"],
        },
    }},
]


def fetch_best_submission(user_id, problem_id, platform):
    # Most tests passed wins; on a tie the most recent attempt.
    best = max(SUBMISSIONS, key=lambda s: (s["passed"], s["at"]))
    return {"ok": True, "submission_id": best["id"], "code": best["code"],
            "passed": best["passed"], "total": len(TESTS)}

## Verification

Not a tool the model can call. If the model could decide whether to run the tests, then
"verified" would not mean anything — so the orchestrator runs this after every `propose_fix`.

Two different failures come out of here and the loop treats them differently:

* `ok=True, verdict="wrong_answer"` — the patch is wrong. Useful information for the model.
* `ok=False` — *our* sandbox broke. That says nothing about the patch.

In [ ]:
def run_tests(code, language="python"):
    if language != "python":
        return {"ok": False, "error": f"no runner for {language}"}

    with tempfile.TemporaryDirectory() as tmp:
        src = Path(tmp) / "main.py"
        src.write_text(code)
        passed, failure = 0, None
        for stdin, expected in TESTS:
            try:
                p = subprocess.run([sys.executable, str(src)], input=stdin,
                                   capture_output=True, text=True, timeout=5)
            except (subprocess.TimeoutExpired, OSError) as exc:
                return {"ok": False, "error": f"{type(exc).__name__}: {exc}"}
            if p.returncode != 0:
                failure = failure or {"input": stdin, "kind": "runtime_error", "stderr": p.stderr[-300:]}
            elif p.stdout.strip() == expected:
                passed += 1
            else:
                failure = failure or {"input": stdin, "kind": "wrong_answer",
                                      "expected": expected, "got": p.stdout.strip()[:100]}

    verdict = "accepted" if passed == len(TESTS) else failure["kind"]
    return {"ok": True, "verdict": verdict, "passed": passed, "total": len(TESTS), "failure": failure}

## Memory

A tally of mistake classes per student. The loop is told the top three before it starts
looking, because the same student keeps making the same mistakes.

In [ ]:
MEMORY = STATE / "memory.json"


def remember(user_id, tag):
    data = json.loads(MEMORY.read_text()) if MEMORY.exists() else {}
    data.setdefault(user_id, {})
    data[user_id][tag] = data[user_id].get(tag, 0) + 1
    STATE.mkdir(exist_ok=True)   # the directory can be gone by now; recreate it at write time
    MEMORY.write_text(json.dumps(data, indent=2))


def top_mistakes(user_id, n=3):
    data = json.loads(MEMORY.read_text()) if MEMORY.exists() else {}
    counts = data.get(user_id, {})
    return sorted(counts, key=counts.get, reverse=True)[:n]

## Talking to the model

One wrapper that turns the SDK reply into `{"text": ..., "calls": [...]}`. If the model
emits arguments that are not valid JSON we set `args` to `None` rather than crashing, and
the loop branches on that.

In [ ]:
def chat(messages, tools, model=MODEL):
    r = client.chat.completions.create(model=model, messages=messages,
                                       tools=tools, tool_choice="auto", temperature=0.2)
    m = r.choices[0].message
    calls = []
    for tc in m.tool_calls or []:
        try:
            args = json.loads(tc.function.arguments)
        except json.JSONDecodeError:
            args = None
        calls.append({"id": tc.id, "name": tc.function.name, "args": args})
    return {"text": m.content, "calls": calls}


def scripted(*replies):
    """Stand-in for the model, so the loop can be tested without an API key."""
    queue = list(replies)

    def fake(messages, tools=None, model=None):
        assert queue, "the loop asked for more replies than the script has"
        return queue.pop(0)

    return fake


def reply(*calls):
    return {"text": None, "calls": [{"id": f"c{i}", "name": n, "args": a}
                                    for i, (n, a) in enumerate(calls)]}

## The loop

Four ways out. The step cap is the backstop, not the plan — the one that actually matters is
the repeated-patch check, because a model that resends the same code would otherwise burn
every step learning nothing.

| stop | status |
|---|---|
| tests pass, then a confirmed submission | `submitted` |
| tests pass, human declined the submission | `fixed_not_submitted` |
| step cap | `exhausted` |
| the same patch twice | `stalled` |
| the test runner broke | `tool_error` |

In [ ]:
SYSTEM = """You debug students' competitive-programming submissions.

Find the single defect and fix it with the smallest change that passes every test. Keep the
student's approach and variable names — a rewrite is useless to us even when it is correct,
because the next stage turns your fix into a hint about their own code.

Call fetch_best_submission first, then propose_fix. Every proposal is executed against the
real tests and you get the first failing test back. If a proposal fails, change your
diagnosis; do not resend the same code."""


def is_yes(answer):
    return answer.strip().lower() == "yes"


def ask_human(question):
    return is_yes(input(f"{question}\nType 'yes' to confirm: "))


def run_repair_loop(user_id, problem_id, platform, chat_fn=chat, max_steps=6, confirm=ask_human):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Problem {problem_id} on {platform}, student {user_id}.\n\n"
                                    f"REFERENCE SOLUTION (never quote it)\n{REFERENCE}\n\n"
                                    f"Their usual mistakes: {top_mistakes(user_id) or 'none yet'}"},
    ]
    seen, fix = set(), None

    for step in range(max_steps):
        out = chat_fn(messages, TOOLS)
        if not out["calls"]:
            return {"status": "gave_up", "step": step, "fix": fix}

        messages.append({"role": "assistant", "content": out["text"], "tool_calls": [
            {"id": c["id"], "type": "function",
             "function": {"name": c["name"], "arguments": json.dumps(c["args"])}}
            for c in out["calls"]]})

        for c in out["calls"]:
            def answer(payload):
                messages.append({"role": "tool", "tool_call_id": c["id"], "content": json.dumps(payload)})

            if c["args"] is None:
                answer({"ok": False, "error": "your arguments were not valid JSON, resend"})

            elif c["name"] == "fetch_best_submission":
                answer(fetch_best_submission(**c["args"]))

            elif c["name"] == "propose_fix":
                code = c["args"]["fixed_code"]
                if code in seen:
                    return {"status": "stalled", "step": step, "fix": fix}
                seen.add(code)

                result = run_tests(code)
                if not result["ok"]:
                    # Our sandbox, not their patch. Stop rather than feed noise back.
                    return {"status": "tool_error", "step": step, "error": result["error"], "fix": fix}

                if result["verdict"] == "accepted":
                    fix = {"code": code, "diagnosis": c["args"]["diagnosis"],
                           "tag": c["args"]["mistake_tag"], "original": BUGGY}
                    remember(user_id, c["args"]["mistake_tag"])
                    answer({"ok": True, "verdict": "accepted",
                            "note": "all tests pass — call submit_to_platform if you want it recorded"})
                else:
                    answer({"ok": True, "verdict": result["verdict"], "passed": result["passed"],
                            "total": result["total"], "failure": result["failure"]})

            elif c["name"] == "submit_to_platform":
                if fix is None:
                    answer({"ok": False, "error": "nothing has passed the tests yet"})
                elif not confirm(f"Submit to {c['args']['platform']} problem "
                                 f"{c['args']['problem_id']} under the service account? "
                                 f"This is public and cannot be undone."):
                    return {"status": "fixed_not_submitted", "step": step, "fix": fix}
                else:
                    return {"status": "submitted", "step": step, "fix": fix}

            else:
                answer({"ok": False, "error": f"no tool called {c['name']}"})

    return {"status": "exhausted", "step": max_steps, "fix": fix}

## Tests

The model is scripted, but the tools are not: `run_tests` really does spawn a Python
subprocess every time, so a scripted "bad fix" fails because the code is genuinely wrong.

In [ ]:
passed = failed = 0


def check(label, condition, detail=""):
    global passed, failed
    passed, failed = passed + bool(condition), failed + (not condition)
    print(("  PASS  " if condition else "  FAIL  ") + label, "" if condition else detail)


BAD_FIX = BUGGY.replace("range(1, n - k)", "range(1, n)")        # IndexError
GOOD_FIX = REFERENCE

FETCH = ("fetch_best_submission", {"user_id": "u42", "problem_id": "1729A", "platform": "codeforces"})


def fix_call(code):
    return ("propose_fix", {"diagnosis": "loop stops one window early",
                            "mistake_tag": "off_by_one", "fixed_code": code})


SUBMIT = ("submit_to_platform", {"problem_id": "1729A", "platform": "codeforces"})

In [ ]:
MEMORY.unlink(missing_ok=True)
print("1. the fixture is real")
check("the student's code passes 3 of 5", run_tests(BUGGY)["passed"] == 3, run_tests(BUGGY))
check("the reference passes 5 of 5", run_tests(REFERENCE)["passed"] == 5)
check("the best attempt is the most recent of the two 3/5 ones",
      fetch_best_submission("u42", "1729A", "codeforces")["submission_id"] == "s102")

print("\n2. happy path: one bad patch, then a good one, then a confirmed submit")
r = run_repair_loop("u42", "1729A", "codeforces",
                    chat_fn=scripted(reply(FETCH), reply(fix_call(BAD_FIX)),
                                     reply(fix_call(GOOD_FIX)), reply(SUBMIT)),
                    confirm=lambda q: True)
check("status is submitted", r["status"] == "submitted", r["status"])
check("the fix and the original are both kept for loop 2",
      r["fix"]["code"] == GOOD_FIX and r["fix"]["original"] == BUGGY)
check("the mistake tag went into memory", "off_by_one" in top_mistakes("u42"))

In [ ]:
print("\n3. stopping conditions")
r = run_repair_loop("u42", "1729A", "codeforces",
                    chat_fn=scripted(reply(FETCH), reply(fix_call(BAD_FIX)), reply(fix_call(BAD_FIX))))
check("the same patch twice stops the loop", r["status"] == "stalled", r["status"])

r = run_repair_loop("u42", "1729A", "codeforces",
                    chat_fn=scripted(reply(fix_call(BAD_FIX + "# a")), reply(fix_call(BAD_FIX + "# b"))),
                    max_steps=2)
check("the step cap stops the loop", r["status"] == "exhausted", r["status"])

print("\n4. a broken test runner is its own branch, not a failing test")
# run_tests refuses a language it has no runner for. Feed it a patch that is actually correct:
# the loop must report tool_error, not 'fixed' and not 'wrong answer'.
saved = run_tests
run_tests = lambda code, language="python": {"ok": False, "error": "gcc not found"}
r = run_repair_loop("u42", "1729A", "codeforces", chat_fn=scripted(reply(fix_call(GOOD_FIX))))
run_tests = saved
check("status is tool_error", r["status"] == "tool_error", r["status"])
check("a correct patch was not reported as fixed", r["fix"] is None)

In [ ]:
print("\n5. the approval gate")
r = run_repair_loop("u42", "1729A", "codeforces",
                    chat_fn=scripted(reply(fix_call(GOOD_FIX)), reply(SUBMIT)),
                    confirm=lambda q: False)
check("declining stops the submission", r["status"] == "fixed_not_submitted", r["status"])
check("...but the fix is still available for loop 2", r["fix"]["code"] == GOOD_FIX)
check("only a literal 'yes' opens the gate",
      [is_yes(a) for a in ["yes", " YES ", "y", "yes please", "sure", ""]]
      == [True, True, False, False, False, False])

print(f"\n{passed} passed, {failed} failed")

## Live run

Needs `LLM_BASE_URL` and `LLM_API_KEY` in `.env`. Everything above runs without them.

In [ ]:
if not OFFLINE:
    r = run_repair_loop("u42", "1729A", "codeforces")
    print(r["status"])
    if r["fix"]:
        print(r["fix"]["diagnosis"], "|", r["fix"]["tag"])
        print(r["fix"]["code"])
        (STATE / "loop1_result.json").write_text(json.dumps(r["fix"], indent=2))
else:
    print("offline — set LLM_API_KEY in .env for this cell")